# 📊 Avaliação de Acurácia e Eficiência do SRI

Este notebook contém a implementação para avaliar a **Eficácia (Qualidade)** e a **Eficiência (Performance)** do Sistema de Recuperação de Informação (SRI) baseado em TF-IDF e Similaridade de Cosseno do RecomendAI.

In [1]:
import sys
import os
import time
import numpy as np
import pandas as pd

# Ajustar caminho para importar os módulos do projeto
sys.path.append(os.path.abspath(os.path.join('..')))

In [2]:
from models.recommendation import MovieRecommender

# Inicializar o recomendador
recommender = MovieRecommender(data_path="../data/tmdb_movies_large.json", weights_path="../models/weights")

✅ Modelo TF-IDF carregado com sucesso!
✅ Modelo SVD carregado com sucesso!


## 🧪 Função de Avaliação do SRI

A função a seguir avalia:
1. **Eficácia (Qualidade)**: Calculando a **Precisão em K (Precision@K)** com base no compartilhamento de pelo menos um gênero entre o filme de origem e as recomendações.
2. **Eficiência (Performance)**: Medindo o **Tempo Médio de Resposta (Latência)** por consulta em milissegundos.

In [3]:
def avaliar_sri(recommender, k=10, amostra_tamanho=100):
    tempos_execucao = []
    precisoes = []
    
    # Filtrar apenas filmes com gêneros válidos cadastrados
    filmes_validos = [m for m in recommender.movies if m.get('genres')]
    
    # Amostrar filmes aleatoriamente de forma determinística
    np.random.seed(42)
    amostra = np.random.choice(filmes_validos, min(amostra_tamanho, len(filmes_validos)), replace=False)
    
    print(f"Iniciando avaliação para {len(amostra)} filmes com K={k}...\n")
    
    for filme_origem in amostra:
        id_origem = filme_origem['id']
        generos_origem = set(filme_origem['genres'])
        
        # Medir latência de resposta
        inicio = time.perf_counter()
        recomendacoes = recommender.get_content_recommendations(id_origem, n=k)
        fim = time.perf_counter()
        tempos_execucao.append(fim - inicio)
        
        # Calcular Precision@K
        relevantes_recuperados = 0
        for rec in recomendacoes:
            filme_rec = next((m for m in recommender.movies if m['id'] == rec['movie_id']), None)
            if filme_rec and filme_rec.get('genres'):
                generos_rec = set(filme_rec['genres'])
                if generos_origem.intersection(generos_rec):
                    relevantes_recuperados += 1
                    
        precisao_k = relevantes_recuperados / k
        precisoes.append(precisao_k)
        
    precisao_media = np.mean(precisoes)
    latencia_media = np.mean(tempos_execucao) * 1000  # converter para ms
    
    print("=" * 50)
    print(f"📊 RESULTADOS DA AVALIAÇÃO DO SRI:")
    print(f"  • Précisão Média em K={k}: {precisao_media:.2%}")
    print(f"  • Tempo de Resposta Médio: {latencia_media:.2f} ms")
    print("=" * 50)
    
    return precisao_media, latencia_media

## 🏃 Executar Métricas

Vamos executar o teste para diferentes valores de K (por exemplo: K=5, K=10 e K=15) para observar o comportamento da precisão e do tempo de resposta.

In [4]:
for k_val in [5, 10, 15]:
    avaliar_sri(recommender, k=k_val, amostra_tamanho=100)

Iniciando avaliação para 100 filmes com K=5...

📊 RESULTADOS DA AVALIAÇÃO DO SRI:
  • Précisão Média em K=5: 88.40%
  • Tempo de Resposta Médio: 1.86 ms
Iniciando avaliação para 100 filmes com K=10...

📊 RESULTADOS DA AVALIAÇÃO DO SRI:
  • Précisão Média em K=10: 86.80%
  • Tempo de Resposta Médio: 1.59 ms
Iniciando avaliação para 100 filmes com K=15...

📊 RESULTADOS DA AVALIAÇÃO DO SRI:
  • Précisão Média em K=15: 85.13%
  • Tempo de Resposta Médio: 1.60 ms
